# Bronze layer

**What this notebook produces:** a working, idempotent bronze table, plus the numbers
that justify every decision in [`docs/bronze.md`](../docs/bronze.md).

Order of business:

1. Look at what the source actually gave us
2. Write down the schema contract
3. Build the schema check *(the one thing you write yourself)*
4. Add provenance and partition columns
5. Predict the output file count, then write, then check the prediction
6. **Append the same month twice and watch the count double.** Once. Then delete it
7. Rebuild correctly with dynamic partition overwrite and prove it is idempotent

Keep the Spark UI open at http://localhost:4040 the whole time.

## 0. Setup

In [ ]:
import sys, shutil
from pathlib import Path
from datetime import datetime, timezone

sys.path.insert(0, str(Path.cwd().parent))   # so `import src.…` works from notebooks/

from pyspark.sql import functions as F
from src.session import get_spark
from src import config

spark = get_spark("bronze")
sc = spark.sparkContext

print("spark", spark.version, "| cores:", sc.defaultParallelism)
print("raw   :", config.RAW_YELLOW)
print("bronze:", config.BRONZE)
print("months found:", config.available_months())

## 1. What did the source give us?

Bronze's whole job is to absorb a boundary you do not control. So the first move is
always the same: **look at the thing before you touch it.**

In [ ]:
path = config.raw_month_path(2024, 3)
df = spark.read.parquet(str(path))

print("input partitions:", df.rdd.getNumPartitions())
print("columns         :", len(df.columns))
print()
for f in df.schema.fields:
    print(f"  {f.name:<24} {f.dataType.simpleString()}")

Neither of those read a single row of data. Parquet carries its schema in a **footer**
at the end of the file, so `printSchema` reads a few kilobytes. That is the difference
between Parquet and CSV in one sentence, and it is why the schema check below is cheap.

### Predict before you run the next cell

You counted March on day 1. Write the number down, then run it.

In [ ]:
sc.setJobDescription("bronze | raw count 2024-03")
n_raw = df.count()
print(f"{n_raw:,}")

## 2. Write the contract down

"Expected schema" has to live **outside the data**, or "expected" just means "whatever
arrived", and a check that compares the file to itself always passes.

The cell below prints a dict you paste into the next cell. This is the only time it is
generated from data. From here on it is a constant that a human edits deliberately.

In [ ]:
print("EXPECTED = {")
for f in df.schema.fields:
    print(f'    "{f.name}": "{f.dataType.simpleString()}",')
print("}")

## 3. The schema check ⭐ you write this one

This is the piece of logic worth twenty minutes, because it is the whole reason bronze
exists as a layer. Everything else in this notebook is plumbing.

**The rules, from `docs/bronze.md` §5:**

| Change at the source | Do |
|---|---|
| New column appears | **Accept**, print it loudly |
| Expected column missing | **Raise** |
| Type changed | **Raise** |
| Column order changed | Ignore. We never read by position |

The three set expressions you need are already written for you. What is left is
deciding what to do with them, and writing an error message that a person woken at 3am
can act on. `ValueError("schema mismatch")` is a useless error. Name the columns and
name the types.

In [ ]:
EXPECTED = {
    # paste the output of the previous cell here
}


def check_schema(df, expected=EXPECTED):
    '''Fail loudly if an incoming file does not match the contract.'''
    actual = {f.name: f.dataType.simpleString() for f in df.schema.fields}

    missing = set(expected) - set(actual)
    added   = set(actual) - set(expected)
    retyped = {c for c in set(expected) & set(actual) if expected[c] != actual[c]}

    # YOUR TURN
    #  - `added`   -> print a loud warning, carry on
    #  - `missing` -> raise, and say which columns
    #  - `retyped` -> raise, and say the expected type AND the type that arrived
    ...

**The test.** Run this once you think it works. Four cases, three of them should not pass silently.

In [ ]:
def expect(label, fn, should_raise):
    try:
        fn()
        print(f"{'FAIL' if should_raise else 'ok  '}  {label}")
    except Exception as e:
        print(f"{'ok  ' if should_raise else 'FAIL'}  {label}  ->  {type(e).__name__}: {e}")

expect("unchanged        ", lambda: check_schema(df), should_raise=False)
expect("column added     ", lambda: check_schema(df.withColumn("surge_fee", F.lit(1.0))), should_raise=False)
expect("column missing   ", lambda: check_schema(df.drop("passenger_count")), should_raise=True)
expect("column retyped   ", lambda: check_schema(df.withColumn("trip_distance", F.col("trip_distance").cast("string"))), should_raise=True)

## 4. Provenance and partition columns

Five added columns. Three say where the row came from, two say where it goes on disk.

Note what `year` and `month` are: **literals from the filename**, not anything derived
from `tpep_pickup_datetime`. The March file contains trips dated 2009. Deciding those
are wrong is a judgement, and judgements live in silver.

In [ ]:
year, month = 2024, 3
batch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

bronze_df = (
    df
    .withColumn(config.COL_SOURCE_FILE, F.input_file_name())
    .withColumn(config.COL_INGESTED_AT, F.current_timestamp())
    .withColumn(config.COL_BATCH_ID,    F.lit(batch_id))
    .withColumn("year",  F.lit(year))
    .withColumn("month", F.lit(month))
)

print("batch:", batch_id)
(bronze_df
   .select(config.COL_SOURCE_FILE, config.COL_INGESTED_AT, config.COL_BATCH_ID, "year", "month")
   .show(3, truncate=False))

Check the Spark UI. **Five `withColumn` calls produced zero jobs.** `show(3)` produced
one. That is lazy evaluation doing its job, and it is why you never optimise by
counting lines of transformation code.

### Predict the output file count

No shuffle happens in this write, so the number of output files per partition equals
the number of Spark partitions holding that data. You have the formula from day 2:

```
totalBytes    = fileSize + openCostInBytes
bytesPerCore  = totalBytes / defaultParallelism
maxSplitBytes = min(maxPartitionBytes, max(openCostInBytes, bytesPerCore))
```

March is 57.3 MB, `openCostInBytes` is 4 MiB, and you know your core count. Work out
how many files you expect in `year=2024/month=3/`, write it down, then run the write.

## 5. Write it

In [ ]:
out = str(config.BRONZE)

sc.setJobDescription("bronze | write 2024-03")
(bronze_df.write
    .mode("append")
    .partitionBy("year", "month")
    .parquet(out))

for p in sorted(Path(out).rglob("*")):
    if p.is_file() and not p.name.startswith("_"):
        print(f"{p.stat().st_size/1e6:8.1f} MB   {p.relative_to(out)}")

Look at the paths. `year=2024/month=3/` is **one partition**, a folder. The `part-*`
files inside it are just files, one per writing task. Those are two different levels and
conflating them is the most common Spark misreading there is.

Also note `month=3`, not `month=03`. Add a tenth month and directory listings sort as
1, 10, 11, 12, 2, 3. It works, it just reads badly. Decide now whether you care.

In [ ]:
back = spark.read.parquet(out)
print(f"rows      {back.count():,}   (source was {n_raw:,})")
print(f"columns   {len(back.columns)}   = 19 source + 3 provenance + 2 partition")
back.select("year", "month").distinct().show()

**`year` and `month` came back, but they are not in the Parquet files.** Spark stripped
them out when writing, because the directory path already encodes the value, and
reconstructed them from the path on read. That is why a partition column costs no
storage no matter how long its name is.

---

## 6. The experiment: run it twice

This is the whole reason approach A is wrong. Do it once, see the number, delete it.

In [ ]:
sc.setJobDescription("bronze | write 2024-03 AGAIN (append)")
(bronze_df.write
    .mode("append")
    .partitionBy("year", "month")
    .parquet(out))

print(f"after second append: {spark.read.parquet(out).count():,}   (should be {n_raw:,})")

No error. No warning. Every downstream number is now inflated by exactly 100% and
nothing in the system knows.

That is the failure mode this layer exists to prevent, and it is why "the job succeeded"
is not the same claim as "the data is right".

Clear it and build it properly.

In [ ]:
shutil.rmtree(out, ignore_errors=True)
print("cleared:", out)

## 7. The correct version

One config line changes the meaning of `mode("overwrite")`:

| `partitionOverwriteMode` | What `overwrite` does |
|---|---|
| `static` **(the default)** | Deletes **the entire table**, writes only the incoming partitions |
| `dynamic` | Replaces only the partitions present in the incoming data |

The default is the most destructive default in Spark. Re-running one month with
`static` silently drops the other eleven.

In [ ]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")


def land_month(year: int, month: int, batch_id: str) -> None:
    src = spark.read.parquet(str(config.raw_month_path(year, month)))
    check_schema(src)

    (src
        .withColumn(config.COL_SOURCE_FILE, F.input_file_name())
        .withColumn(config.COL_INGESTED_AT, F.current_timestamp())
        .withColumn(config.COL_BATCH_ID,    F.lit(batch_id))
        .withColumn("year",  F.lit(year))
        .withColumn("month", F.lit(month))
        .write
        .mode("overwrite")
        .partitionBy("year", "month")
        .parquet(str(config.BRONZE)))


def run_all() -> str:
    batch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    for y, m in config.available_months():
        sc.setJobDescription(f"bronze | {y}-{m:02d} | batch {batch_id}")
        land_month(y, m, batch_id)
    return batch_id

In [ ]:
b1 = run_all()
print(f"run 1  batch={b1}  rows={spark.read.parquet(str(config.BRONZE)).count():,}")

b2 = run_all()
print(f"run 2  batch={b2}  rows={spark.read.parquet(str(config.BRONZE)).count():,}")

**Same number twice. That is idempotency**, and it is a property of the *result*, not of
the run. A job that succeeds twice and doubles the rows also succeeded twice.

What it is still not: **atomic**. Kill the kernel halfway through that write and you get
a half-replaced partition, because the delete and the write are separate steps. That is
the gap Iceberg closes, and it is the honest reason to reach for it here rather than
"Iceberg is the modern way".

In [ ]:
(spark.read.parquet(str(config.BRONZE))
   .groupBy("year", "month", config.COL_BATCH_ID)
   .count()
   .orderBy("year", "month")
   .show(truncate=False))

Only the second batch id survives. That is the proof the write **replaced** rather than
appended, and it is the observability property earning its keep: you can tell which run
produced every row in the table.

---

## 8. Fill this in before moving on

| Question | Predicted | Actual |
|---|---|---|
| Rows in March | | |
| Files in `year=2024/month=3/` | | |
| Bronze size on disk vs the 160 MB source | | |
| Rows after the double append | | |
| Rows after two correct runs | | |

Any row where predicted and actual disagree is the only interesting row on the page.
Put the reason in `docs/bronze.md` §10.

## 9. What goes into `src/bronze/ingest.py`

Four things, lifted straight out of this notebook:

- `EXPECTED` and `check_schema()` → their own module, `src/bronze/schema.py`
- `land_month(year, month, batch_id)` → unchanged
- `run_all()` → renamed `main()`, so Airflow can call it as one task
- the `partitionOverwriteMode` config → moves into `src/session.py`, so no caller can
  forget it and trigger the static-overwrite footgun

Then `python -m src.bronze.ingest` twice, and the row count does not move.